# Najafi et al. (2020) on DANDI:000016, notebook 1: behaviour, single neurons and correlations

**Paper.** Najafi F, Elsayed GF, Cao R, Pnevmatikakis E, Latham PE, Cunningham JP, Churchland AK (2020).
Excitatory and inhibitory subnetworks are equally selective during decision-making and emerge
simultaneously during learning. *Neuron* 105, 165-179. [doi:10.1016/j.neuron.2019.09.045](https://doi.org/10.1016/j.neuron.2019.09.045)

**Dataset.** [DANDI:000016](https://dandiarchive.org/dandiset/000016/0.260915.2309), version
0.260915.2309 (doi:10.48324/dandi.000016/0.260915.2309), streamed from the archive. This is the only
published version (2026-09-15); the paper cites no DANDI version (it names the NWB deposit at the
CSHL repository, doi:10.14224/1.37693, from which the dandiset is derived), and the draft holds
the same 135 files.

**Claims.** `claims/000016/10.1016-j.neuron.2019.09.045.yaml`, written from the accepted author
manuscript in PubMed Central (PMC6952547) with its supplement, and the paper's code
(github.com/farznaj/imaging_decisionMaking_exc_inh at commit b7352acd). The claims file decides what
is tested and how; this notebook does not change a claim.

The entry has eight notebooks, each under the 30-minute limit; this one holds the data check of
the whole dandiset and the claims on behaviour, single neurons and pairwise correlations. All 38
claims and where each is tested:

| Claim | Finding | Notebook | Status |
|---|---|---|---|
| `data-composition` | 135 sessions from 4 mice; trials, neurons and inhibitory fraction | 01 | tested |
| `fig1b-psychometric` | Choices depend on stimulus rate | 01 | tested |
| `figs1b-trial-history` | Choice depends on stimulus, previous outcome and ITI | 01 | blocked (data missing: no inter-trial intervals) |
| `fig1gi-inferred-spiking-differs` | Inferred spiking differs between E and I | 01 | tested |
| `fig2a-example-session` | Example session, 24 E and 5 I neurons selective | 01 | tested |
| `fig2bc-fraction-selective-similar` | Similar fraction of choice-selective E and I neurons | 01 | tested |
| `fig2bd-selectivity-time-course` | Selectivity grows over the trial, peaks just after the choice | 01 | tested |
| `fig2de-selectivity-magnitude-similar` | Similar magnitude of selectivity in E and I | 01 | tested |
| `figs3d-same-spiking-selectivity` | Similar selectivity for activity-matched neurons | 01 | tested |
| `figs3c-wider-window` | Fractions selective in a wider window | 01 | tested |
| `figs3e-correct-error-correlation` | Correct and error selectivity correlate, similarly for E and I | 01 | tested |
| `fig3bc-decoding-similar` | E and I populations decode choice equally well | 02 | tested |
| `fig3b-decoding-time-course` | All-neuron accuracy grows and peaks at the choice | 03 | tested |
| `fig3df-weights-similar` | Matched decoder weights of E and I | 02 (bin before choice), 03 (every bin) | tested |
| `fig3hi-mixed-no-advantage` | Mixed E and I population no better than E | 05 | tested |
| `figs5-fluorescence-decoding` | Equal decoding on fluorescence traces | 01 | blocked (data missing: no fluorescence traces) |
| `figs6ab-previous-choice-stimulus` | Previous choice and stimulus category decoding | 06 | tested |
| `figs6c-outcome-decoding` | Outcome decoding reaches 80% | 05 | tested |
| `figs6de-incorrect-trials` | Decoders generalise to incorrect trials similarly for E and I | 02 | tested |
| `fig4bd-stability-similar` | Similar stability of E and I decoders | 02 | tested |
| `figs7c-angle-stability` | Similar angle-based stability | 04 | tested |
| `fig5a-nonselective-model` | Non-selective model: I below E | 08 | tested (simulation) |
| `fig5bc-selective-models-match` | Selective models: E and I match near ratio 1 | 08 | tested (simulation) |
| `figs9-signal-selective-best` | Signal-selective model is most accurate | 08 | tested (simulation) |
| `fig6a-same-selectivity-correlations` | Stronger correlations for same selectivity | 01 | tested |
| `fig6bc-inhibitory-correlations-higher` | II correlations above EE, EI between | 01 | tested |
| `fig7-noise-correlations-limit-decoding` | Pseudo populations decode better | 05 | tested |
| `fig8ab-learning-accuracy` | Accuracy rises with learning and tracks performance | 07a, 07b, 07c | tested |
| `fig8ce-choice-signal-onset` | Choice signal onset moves earlier with learning | 07a, 07b, 07c | tested |
| `fig8de-learning-parallel` | E and I matched through learning | 07a, 07b, 07c | tested |
| `figs13c-equal-trials` | Learning increase with equal trial numbers | 07a, 07b, 07c | tested |
| `figs12-licking-running` | Licking and running do not explain learning | 07a, 07b, 07c | blocked in part (lick spout not identified); running part tested |
| `fig8f-correlations-decrease` | Noise correlations fall with learning | 01 | tested |
| `figs13ab-pseudo-learning` | Pseudo-population accuracy rises with learning | 07a, 07b, 07c | tested |
| `fig8g-fraction-selective-triples` | Fraction selective triples with learning | 01 | tested |
| `figs2-temporal-dynamics-similar` | Similar peak-time distributions of E and I | 01 | tested |
| `figs14-neuropil` | CNMF and annulus traces agree (r = 0.96) | 01 | blocked (data missing: no raw movies) |
| `figs15-labeling-specificity` | tdTomato and GABA co-labelling | 01 | blocked (data missing: no histology) |

The decoding claims need many support vector machines per session; notebooks 2 to 7 run them, and
notebook 8 runs the circuit model. Learning claims are judged per mouse in notebooks 7a to 7c and
combined in the attempt record.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

import najafi as nj
import sessions as S

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
MICE, LEARN = [1, 2, 3, 4], [1, 2, 4]

A = nj.asset_urls()
print(f"DANDI:{nj.DANDISET_ID} version {nj.DANDISET_VERSION}: {len(A)} NWB files, {A['size'].sum() / 1e9:.1f} GB")
print(A.groupby("subject").size())
items = [(i, p, u) for i, (p, u) in enumerate(zip(A.path, A.url))]
R = nj.map_sessions(S.single_neurons, items, workers=4)
sess = pd.DataFrame([{k: r[k] for k in ("idx", "path", "mouse", "date", "description", "imaging_rate", "period",
                                         "n_trials", "n_decided", "n_correct", "n_incorrect", "n_choice_valid",
                                         "n_good", "n_bad", "n_traced", "traced_all_good", "n_exc", "n_inh", "n_unk",
                                         "bad_typed", "performance", "roc_ok", "trial_hash", "event_order_ok",
                                         "lick_at_choice", "lick_step", "wheel_step", "min_value", "max_value",
                                         "frac_near_zero")} for r in R])
print(len(sess), "sessions loaded")

## Data check

What was loaded, and each description checked against the values. The trial table, ROI table and
event-aligned traces of every session are read; the neural traces enter as non-overlapping 3-frame
bins counted from each event (najafi.py).

In [ ]:
by = sess.groupby("mouse")
print("Sessions, recording dates and counts per mouse:")
print(by.agg(sessions=("path", "size"), first_day=("date", "min"), last_day=("date", "max"),
             distinct_days=("date", "nunique"), trials_median=("n_trials", "median"),
             with_choice_median=("n_decided", "median"), correct_mean=("n_correct", "mean"),
             incorrect_mean=("n_incorrect", "mean"), good_rois_mean=("n_good", "mean"),
             exc_mean=("n_exc", "mean"), inh_mean=("n_inh", "mean"), unknown_mean=("n_unk", "mean")).round(1).to_string())
print()
print("Recordings deposited twice (identical trial tables):", int(sess.trial_hash.duplicated().sum()))
print("Sessions per (mouse, day) above 1:", int(sess.duplicated(["mouse", "date"]).sum()))
print("Every traced ROI has status good:", bool(sess.traced_all_good.all()),
      "; traced ROIs = good ROIs in every session:", bool((sess.n_traced == sess.n_good).all()))
print("Bad ROIs labelled excitatory or inhibitory:", int(sess.bad_typed.sum()))
print()
print("Imaging rate in the metadata:", sorted(sess.imaging_rate.unique()), "Hz")
print("Frame period stored with the traces: %.5f to %.5f s, i.e. %.2f Hz; 3-frame bin = %.1f ms"
      % (sess.period.min(), sess.period.max(), 1 / sess.period.mean(), 3000 * sess.period.mean()))
st = np.array([r["start_time_range"] for r in R]); sp = np.array([r["stop_time_range"] for r in R])
print("Trial start_time over all sessions: %.3f to %.3f s; stop_time %.2f to %.2f s (times are relative to each trial)"
      % (st.min(), st.max(), sp.min(), sp.max()))
rr = np.array([r["rate_range"] for r in R])
print("Stimulus rate: %.2f to %.2f Hz; trials at exactly 16 Hz: %d"
      % (rr.min(), rr.max(), sum(int(r["behaviour"].is16.sum()) for r in R)))
resp = pd.DataFrame([r["responses"] for r in R]).fillna(0).sum().astype(int)
print("Trial outcomes over all sessions:", resp.to_dict())
print("Decided trials whose events are in order (init < stim on < stim off, go < choice): %.4f" % sess.event_order_ok.mean())
print()
print("Trace values (all sessions, choice-aligned): min %.2e, max %.3f, fraction within 1e-6 of zero %.3f"
      % (sess.min_value.min(), sess.max_value.max(), sess.frac_near_zero.mean()))
mx = np.concatenate([r["neuron_max_any_alignment"] for r in R])
print("Per-neuron maximum over the stored alignments: %.1f%% of %d neurons have exactly 1 (to 1e-6), min %.3f"
      % (100 * np.mean(np.abs(mx - 1) < 1e-6), len(mx), np.nanmin(mx)))
print()
fr = pd.DataFrame([{k: v[0] for k, v in r["frames"].items()} for r in R])
ev = pd.DataFrame([{k: v[1] for k, v in r["frames"].items()} for r in R])
print("Frames before each event (min to max over sessions):", {k: (int(ev[k].min()), int(ev[k].max())) for k in ev})
print("Frames from each event on:", {k: (int((fr[k] - ev[k]).min()), int((fr[k] - ev[k]).max())) for k in fr})
print()
print("Lick trace: values", sorted(set(v for r in R for v in r["lick_values"])), "; step %.4f s" % sess.lick_step.median())
print("Fraction of choice-aligned trials with a lick within 1.5 ms of time 0: %.3f (min over sessions %.3f)"
      % (sess.lick_at_choice.mean(), sess.lick_at_choice.min()))
wr = np.array([r["wheel_range"] for r in R])
print("Wheel trace: step %.3f s, values %.3f to %.3f (unit 'unknown' in the file)" % (sess.wheel_step.median(), wr.min(), wr.max()))
pre = np.array([r.get("sig_pre_init", np.nan) for r in R])
at = np.array([r["roc"]["exc"]["sig"][r["i1"]].mean() if r["roc_ok"] else np.nan for r in R])
print("Timing contrast: fraction of excitatory neurons choice-selective 0-97 ms before the initiation tone %.3f, "
      "0-97 ms before the choice %.3f (mean over sessions)" % (np.nanmean(pre), np.nanmean(at)))
print("Processing modules in the files:", sorted(set(k for r in R for k in r["available"])))

Checks of the description against the data:

- The 135 files hold 45, 48, 7 and 35 sessions of four subjects named mouse1_fni16 to mouse4_fni19,
  one session per recording day; no trial table appears twice.
- The ROI table marks each ROI good or bad and excitatory, inhibitory or unknown; the traces hold
  every good ROI and only those. "Unknown" is the paper's "unsure" class.
- The groups named `dFoF_*` do not hold Delta F/F: their values are non-negative to numerical
  precision, mostly zero, and at most 1. This is the
  inferred spiking activity normalised to a maximum of 1 that the paper analyses (STAR Methods);
  about half of the neurons reach exactly 1 within the stored windows (the maximum can fall
  outside them) and none exceeds 1. Recorded as a metadata error.
- The metadata give an imaging rate of 30 Hz; the frame period stored with the traces is 32.4 ms
  (30.9 Hz), as the paper's 97 ms bins of 3 frames imply. The traces store the period in the
  attribute `rate`. Recorded as a metadata error.
- Trial start and stop times are relative to each trial (start_time near 0), so the time between
  trials cannot be recovered.
- The traces are aligned by the authors to the trial events; the lick trace has a lick at time 0
  of the choice-aligned window in nearly every trial, and choice selectivity is near chance before
  the initiation tone and higher just before the choice. These are consistent with the traces
  sharing the clock of the trial events, but do not prove it for every stream.

## data-composition

> The imaging dataset has 135 sessions from 4 mice (45, 48, 7 and 35 sessions), with on average 480
> neurons per session, about 40 inhibitory, 330 excitatory and 100 unclassified; inhibitory neurons
> are 11% of the population.

Criterion: mice matched by session counts; neurons are good ROIs; trials counted (a) as all trials
of the trial table and (b) as trials with a correct or incorrect choice; the inhibitory fraction (a)
over all neurons and (b) over excitatory plus inhibitory. Reproduced when every value is within
tolerance under one reading of each.

In [ ]:
exp = {"sessions": 135, "median trials, mouse 1": 213, "median trials, mouse 2": 253, "median trials, mouse 3": 264,
       "median trials, mouse 4": 222, "neurons per session": 480, "inhibitory per session": 40,
       "excitatory per session": 330, "inhibitory fraction": 0.11}
tol = {k: (0.1 * v) for k, v in exp.items()}
rows = []
rows.append(("sessions", "", len(sess)))
print("Sessions per mouse:", sess.groupby("mouse").size().to_dict(), "(paper: 45, 48, 7, 35)")
for m in MICE:
    d = sess[sess.mouse == m]
    rows.append((f"median trials, mouse {m}", "a: all trials", d.n_trials.median()))
    rows.append((f"median trials, mouse {m}", "b: with a choice", d.n_decided.median()))
rows.append(("neurons per session", "", sess.n_good.mean()))
rows.append(("inhibitory per session", "", sess.n_inh.mean()))
rows.append(("excitatory per session", "", sess.n_exc.mean()))
rows.append(("inhibitory fraction", "a: of all neurons", sess.n_inh.sum() / sess.n_good.sum()))
rows.append(("inhibitory fraction", "b: of E + I", sess.n_inh.sum() / (sess.n_inh.sum() + sess.n_exc.sum())))
t = pd.DataFrame(rows, columns=["quantity", "reading", "measured"])
t["expected"] = t.quantity.map(exp); t["tolerance"] = t.quantity.map(tol)
t["within"] = (t.measured - t.expected).abs() <= t.tolerance
print(t.round(4).to_string(index=False))
print("Unclassified (unknown) per session: %.1f" % sess.n_unk.mean())
print("Mean of per-session inhibitory fractions: (a) %.4f, (b) %.4f"
      % ((sess.n_inh / sess.n_good).mean(), (sess.n_inh / (sess.n_inh + sess.n_exc)).mean()))
ok_any = t.groupby("quantity").within.any()
print("Every quantity within tolerance under at least one reading:", bool(ok_any.all()))
print("Reading (a) of trials within tolerance for all mice:", bool(t[t.reading == "a: all trials"].within.all()),
      "; reading (b):", bool(t[t.reading == "b: with a choice"].within.all()))

**Result: (to be written)**

## fig1b-psychometric

> Choices depend strongly on stimulus rate, at chance at the 16 Hz boundary and more accurate for
> rates further from it.

Criterion: per mouse, all trials with a choice from all sessions; fraction of "high" choices per
rate and a logistic regression of the high choice on rate; reproduced when the slope is positive
at p < 0.05 in every mouse. Only the four imaged mice are in the dandiset (the figure uses 10).

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
res = []
for k, m in enumerate(MICE):
    b = pd.concat([r["behaviour"] for r in R if r["mouse"] == m])
    b = b[b.decided]
    X = sm.add_constant(b.trial_pulse_rate.values)
    fit = sm.Logit(b.choice_high.values, X).fit(disp=0)
    at16 = b[b.is16].choice_high.mean()
    res.append(dict(mouse=m, trials=len(b), slope_per_Hz=fit.params[1], p=fit.pvalues[1], intercept=fit.params[0],
                    high_at_16Hz=at16, n_16Hz=int(b.is16.sum()),
                    high_at_5_8Hz=b[b.trial_pulse_rate <= 8].choice_high.mean(),
                    high_at_24_27Hz=b[b.trial_pulse_rate >= 24].choice_high.mean()))
    g = b.groupby(b.trial_pulse_rate.round()).choice_high.agg(["mean", "size"])
    ax[k].plot(g.index, g["mean"], "o", ms=3)
    rr = np.linspace(5, 27, 50)
    ax[k].plot(rr, fit.predict(sm.add_constant(rr)), "k-")
    ax[k].axvline(16, ls="--", c="gray"); ax[k].set_title(f"mouse {m}"); ax[k].set_xlabel("stimulus rate (Hz)")
ax[0].set_ylabel("fraction high choice"); plt.show()
res = pd.DataFrame(res)
print(res.to_string(index=False))
print("All slopes positive with p < 0.05:", bool(((res.slope_per_Hz > 0) & (res.p < 0.05)).all()))
print("Plotted fractions lie in [0, 1]; rates 5 to 27 Hz.")

**Result: (to be written)**

## figs1b-trial-history (blocked)

> Choice depends on current stimulus strength, on the previous trial's outcome and on the time
> since the previous trial; the effect of a previous success is stronger after a short ITI, and
> ITI has no significant effect after a failure.

The criterion splits previous outcomes at an inter-trial interval of 7 s, the time from the previous
choice to the current stimulus onset. The trial tables give event times relative to the start of
each trial only (data check above: start_time within half a second of 0 in every trial), so the
interval between trials is not in the dandiset.

In [ ]:
d = R[0]["behaviour"]
print("start_time of all trials, all sessions: %.3f to %.3f s" % (st.min(), st.max()))
print("No column of the trial table holds an absolute time; columns:", list(d.columns))

## fig1gi-inferred-spiking-differs

> Inferred spiking activity differs significantly between inhibitory and excitatory neurons
> throughout the trial, and in the bin just before the choice in every mouse.

Criterion: correct trials, the bin 0-97 ms before the choice, activity averaged over trials per
neuron; (a) unpaired two-tailed t-test across days of the day means, (b) the same on neurons pooled
over days; direction reported. Reproduced when p < 0.001 in every mouse, and for the time course in
every choice-aligned bin of at least one mouse under reading (a). Expected pooled counts 41,723
excitatory and 5,142 inhibitory neurons.

In [ ]:
rows = []
for m in MICE:
    rr_ = [r for r in R if r["mouse"] == m]
    de = np.array([np.nanmean(r["act_bins"]["exc"][r["i1"]]) for r in rr_])
    di = np.array([np.nanmean(r["act_bins"]["inh"][r["i1"]]) for r in rr_])
    ne = np.concatenate([r["act_bins"]["exc"][r["i1"]] for r in rr_])
    ni = np.concatenate([r["act_bins"]["inh"][r["i1"]] for r in rr_])
    ta = nj.ttest(de, di); tb = nj.ttest(ne, ni)
    rows.append(dict(mouse=m, days=len(de), exc_day_mean=np.nanmean(de), inh_day_mean=np.nanmean(di),
                     p_a_days=ta[1], neurons_exc=tb[2], neurons_inh=tb[3], exc_neuron_mean=np.nanmean(ne),
                     inh_neuron_mean=np.nanmean(ni), p_b_neurons=tb[1]))
t = pd.DataFrame(rows)
print(t.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("Inhibitory higher than excitatory in every mouse:", bool((t.inh_day_mean > t.exc_day_mean).all()))
print("p < 0.001 in every mouse: (a)", bool((t.p_a_days < 0.001).all()), " (b)", bool((t.p_b_neurons < 0.001).all()))
print("Pooled neurons with a value: excitatory %d (expected 41723 +/- 4172), inhibitory %d (expected 5142 +/- 514)"
      % (t.neurons_exc.sum(), t.neurons_inh.sum()))
# time course, reading (a)
print()
for m in MICE:
    rr_ = [r for r in R if r["mouse"] == m]
    de, off = nj.align_bins([np.nanmean(r["act_bins"]["exc"], 1) for r in rr_], [r["i1"] for r in rr_])
    di, _ = nj.align_bins([np.nanmean(r["act_bins"]["inh"], 1) for r in rr_], [r["i1"] for r in rr_])
    p = np.array([stats.ttest_ind(de[:, k], di[:, k]).pvalue for k in range(len(off))])
    print(f"mouse {m}: {len(off)} shared bins ({off[0]} to {off[-1]}), p < 0.001 in {int((p < 0.001).sum())}; "
          f"largest p {p.max():.2g}; inhibitory higher in {int((di.mean(0) > de.mean(0)).sum())} bins")

**Result: (to be written)**

## fig2a-example-session

> In an example session (285 excitatory and 29 inhibitory neurons), 24 excitatory (8%) and 5
> inhibitory (17%) neurons were significantly choice selective just before the choice.

Criterion: the session with 285 excitatory and 29 inhibitory neurons, if exactly one has these
counts; the 50-shuffle ROC repeated 20 times; reproduced when the median counts are within
tolerance (24 +/- 2.4 and 5 +/- 0.5).

In [ ]:
ex = sess[(sess.n_exc == 285) & (sess.n_inh == 29)]
print("Sessions with 285 excitatory and 29 inhibitory neurons:", ex[["path", "mouse", "date"]].to_dict("records"))
if len(ex) == 1:
    i = int(ex.idx.iloc[0])
    rep = nj.map_sessions(S.roc_repeats, [(i, A.path[i], A.url[i], 20)], workers=1)[0]
    cnt = rep["counts"]
    print("Correct trials without 16 Hz, per choice:", rep["n_trials"])
    print("Selective counts over 20 repeats (normal-fit rule): excitatory", cnt[:, 0].tolist())
    print("                                                    inhibitory ", cnt[:, 1].tolist())
    print("Median: excitatory %.1f (%.1f%%), inhibitory %.1f (%.1f%%)" % (np.median(cnt[:, 0]), 100 * np.median(cnt[:, 0]) / 285,
                                                                      np.median(cnt[:, 1]), 100 * np.median(cnt[:, 1]) / 29))
    print("Empirical-p variant, median: excitatory %.1f, inhibitory %.1f" % (np.median(cnt[:, 2]), np.median(cnt[:, 3])))

**Result: (to be written)**

## fig2bc-fraction-selective-similar

> Excitatory and inhibitory neurons have a similar fraction of significantly choice-selective
> neurons just before the choice (median across mice 13% and 16%, about 43 excitatory and 6
> inhibitory neurons per session); the difference is significant in mouse 4.

Criterion: per session the fraction of each type selective at 0-97 ms before the choice; per mouse
the mean over sessions, then the median over mice; unpaired two-tailed t-test across sessions per
mouse at p < 0.05. Reproduced when both medians are within tolerance and no significant difference
is found in mice 1, 2 and 3; partially reproduced when only the t-tests hold. Counts per session as
the median across mice of the mean count. Variant: empirical shuffle p.

In [ ]:
RO = [r for r in R if r["roc_ok"]]
print("Sessions with at least 10 correct trials of each choice (no 16 Hz): %d of %d" % (len(RO), len(R)))
def frac_table(key):
    rows = []
    for m in MICE:
        rr_ = [r for r in RO if r["mouse"] == m]
        fe = np.array([r["roc"]["exc"][key][r["i1"]].mean() for r in rr_])
        fi = np.array([r["roc"]["inh"][key][r["i1"]].mean() for r in rr_])
        ce = np.array([r["roc"]["exc"][key][r["i1"]].sum() for r in rr_])
        ci = np.array([r["roc"]["inh"][key][r["i1"]].sum() for r in rr_])
        tt = nj.ttest(fe, fi)
        rows.append(dict(mouse=m, sessions=len(rr_), frac_exc=fe.mean(), frac_inh=fi.mean(), t=tt[0], p=tt[1],
                         count_exc=ce.mean(), count_inh=ci.mean()))
    return pd.DataFrame(rows)
for key, name in (("sig", "normal-fit rule (main)"), ("sig_emp", "empirical shuffle p (variant)")):
    t = frac_table(key)
    print(f"--- {name}")
    print(t.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    print("median across mice: excitatory %.3f (expected 0.13 +/- 0.013), inhibitory %.3f (expected 0.16 +/- 0.016)"
          % (t.frac_exc.median(), t.frac_inh.median()))
    print("counts per session, median across mice: excitatory %.1f (43 +/- 4.3), inhibitory %.1f (6 +/- 0.6)"
          % (t.count_exc.median(), t.count_inh.median()))
    print("significant (p < 0.05) in mice:", t.mouse[t.p < 0.05].tolist())

**Result: (to be written)**

## fig2bd-selectivity-time-course

> The fraction of choice-selective neurons and the magnitude of choice selectivity increase
> gradually over the trial and peak just after the choice.

Criterion: ROC per 97 ms bin of the choice-aligned traces; per mouse the mean over sessions per bin
and type. "Just after" read as (a) the peak in the first bin after the choice (0-97 ms) and (b)
within 300 ms after the choice (bins starting 0 to 291 ms). Reproduced when, in at least the mouse
with 45 sessions and for both measures and both types, the earliest bin is below the bin before the
choice and the peak lies in the window. Bins are those all of a mouse's sessions share.

In [ ]:
period = sess.period.mean()
fig, ax = plt.subplots(2, 4, figsize=(15, 6))
rows = []
for k, m in enumerate(MICE):
    rr_ = [r for r in RO if r["mouse"] == m]
    for j, (meas, label) in enumerate((("sig", "fraction selective"), ("sel", "selectivity 2|AUC-0.5|"))):
        for typ, col in (("exc", "b"), ("inh", "r")):
            a, off = nj.align_bins([r["roc"][typ][meas].mean(1) for r in rr_], [r["i1"] for r in rr_])
            mean = a.mean(0)
            tms = off * 3 * period * 1000
            pk = int(np.argmax(mean))
            rows.append(dict(mouse=m, sessions=len(rr_), measure=meas, type=typ, first_bin_ms=round(tms[0]),
                             first=mean[0], before_choice=mean[off == -1][0], peak=mean[pk], peak_bin_start_ms=round(tms[pk]),
                             rises=mean[0] < mean[off == -1][0], peak_a=off[pk] == 0, peak_b=(off[pk] >= 0) and (tms[pk] < 300)))
            ax[j, k].plot(tms, mean, col)
        ax[j, k].axvline(0, c="gray", ls="--"); ax[0, k].set_title(f"mouse {m}"); ax[j, 0].set_ylabel(label)
plt.show()
t = pd.DataFrame(rows)
print(t.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("Plotted ranges: fraction selective %.3f to %.3f, selectivity %.3f to %.3f" %
      (t[t.measure == "sig"][["first", "peak"]].min().min(), t[t.measure == "sig"].peak.max(),
       t[t.measure == "sel"][["first", "peak"]].min().min(), t[t.measure == "sel"].peak.max()))
for reading in ("peak_a", "peak_b"):
    ok = t[(t.mouse == 1)]
    print(f"Mouse 1, reading {reading[-1]}: rises in all four series: {bool(ok.rises.all())}; peak in window in all four: {bool(ok[reading].all())}")
    print(f"   all mice: series meeting both: {int((t.rises & t[reading]).sum())} of {len(t)}")

**Result: (to be written)**

## fig2de-selectivity-magnitude-similar

> The magnitude and time course of choice selectivity are similar for excitatory and inhibitory
> neurons.

Criterion: selectivity 2|AUC-0.5| from correct trials, averaged over neurons of each type per
session, at 0-97 ms before the choice; unpaired two-tailed t-test across sessions per mouse at
p < 0.05, a claim of absence. Reproduced when no mouse differs; partially reproduced when one does.
For the time course the same test per bin is shown.

In [ ]:
rows = []
for m in MICE:
    rr_ = [r for r in RO if r["mouse"] == m]
    se = np.array([r["roc"]["exc"]["sel"][r["i1"]].mean() for r in rr_])
    si = np.array([r["roc"]["inh"]["sel"][r["i1"]].mean() for r in rr_])
    sh = np.array([r["roc"]["exc"]["sel_shuffled"][r["i1"]].mean() for r in rr_])
    tt = nj.ttest(se, si)
    ae, off = nj.align_bins([r["roc"]["exc"]["sel"].mean(1) for r in rr_], [r["i1"] for r in rr_])
    ai, _ = nj.align_bins([r["roc"]["inh"]["sel"].mean(1) for r in rr_], [r["i1"] for r in rr_])
    pb = np.array([stats.ttest_ind(ae[:, k], ai[:, k]).pvalue for k in range(len(off))])
    rows.append(dict(mouse=m, sessions=len(rr_), sel_exc=se.mean(), sel_inh=si.mean(), diff_inh_minus_exc=si.mean() - se.mean(),
                     sel_shuffled_exc=sh.mean(), t=tt[0], p=tt[1], bins=len(off), bins_p_below_05=int((pb < 0.05).sum())))
t = pd.DataFrame(rows)
print(t.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("Mice with p < 0.05 at 0-97 ms before choice:", t.mouse[t.p < 0.05].tolist())

**Result: (to be written)**

## figs3d-same-spiking-selectivity

> Restricted to excitatory and inhibitory neurons with the same spiking activity, choice
> selectivity is still similar, and the difference in mouse 4 of Figure 2C disappears.

Criterion: per session the median activity over trials at 0-97 ms before the choice, the range cut
into 50 bins, and only neurons of both types in the bin holding most inhibitory neurons kept;
unpaired two-tailed t-tests across sessions per mouse on session-mean selectivity and fraction
selective. Reproduced when no mouse, mouse 4 included, differs. The medians are taken over the ROC
trials (correct, no 16 Hz).

In [ ]:
rows = []
for m in MICE:
    rr_ = [r for r in RO if r["mouse"] == m and r["matched"]["n_exc"] > 0 and r["matched"]["n_inh"] > 0]
    fe = np.array([r["matched"]["exc_sig"].mean() for r in rr_]); fi = np.array([r["matched"]["inh_sig"].mean() for r in rr_])
    se = np.array([r["matched"]["exc_sel"].mean() for r in rr_]); si = np.array([r["matched"]["inh_sel"].mean() for r in rr_])
    t1, t2 = nj.ttest(fe, fi), nj.ttest(se, si)
    rows.append(dict(mouse=m, sessions=len(rr_), matched_exc_mean=np.mean([r["matched"]["n_exc"] for r in rr_]),
                     matched_inh_mean=np.mean([r["matched"]["n_inh"] for r in rr_]), frac_exc=fe.mean(), frac_inh=fi.mean(),
                     p_frac=t1[1], sel_exc=se.mean(), sel_inh=si.mean(), p_sel=t2[1]))
t = pd.DataFrame(rows)
print(t.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("Sessions left out (no matched neuron of one type):", len(RO) - int(t.sessions.sum()))
print("Mice with a significant difference: fraction", t.mouse[t.p_frac < 0.05].tolist(), "; selectivity", t.mouse[t.p_sel < 0.05].tolist())

**Result: (to be written)**

## figs3c-wider-window

> ROC on the window from 200 ms before to 300 ms after the choice gives a larger fraction of
> selective neurons, 21% of excitatory and 27% of inhibitory (median across mice), about 69
> excitatory and 11 inhibitory neurons per session.

Criterion: as fig2bc with activity averaged over the window; readings (a) the bins from 2 before to
3 after the choice bin (194 ms before to 291 ms after) and (b) the frames from 200 ms before to 300
ms after. Reproduced when all four values are within tolerance.

In [ ]:
for w in ("a", "b"):
    rows = []
    for m in MICE:
        rr_ = [r for r in RO if r["mouse"] == m]
        rows.append(dict(mouse=m, frac_exc=np.mean([r["window"][w]["exc"]["sig"].mean() for r in rr_]),
                         frac_inh=np.mean([r["window"][w]["inh"]["sig"].mean() for r in rr_]),
                         count_exc=np.mean([r["window"][w]["exc"]["sig"].sum() for r in rr_]),
                         count_inh=np.mean([r["window"][w]["inh"]["sig"].sum() for r in rr_])))
    t = pd.DataFrame(rows)
    vals = [t.frac_exc.median(), t.frac_inh.median(), t.count_exc.median(), t.count_inh.median()]
    expd = [(0.21, 0.021), (0.27, 0.027), (69, 6.9), (11, 1.1)]
    within = [abs(v - e) <= tl for v, (e, tl) in zip(vals, expd)]
    print(f"--- reading {w}"); print(t.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    print("medians across mice: excitatory %.3f, inhibitory %.3f; counts %.1f and %.1f; within tolerance:" % tuple(vals), within)

**Result: (to be written)**

## figs3e-correct-error-correlation

> Choice selectivity on correct and on error trials is positively correlated across neurons, so
> most neurons reflect the choice more than the stimulus, and within each mouse this correlation is
> very similar for excitatory and inhibitory neurons; it is weaker in mouse 4.

Criterion: per session and neuron, AUC on correct and on error trials (same labels, 0-97 ms before
the choice); Pearson r across neurons per type; per mouse the mean r over sessions with at least 10
error trials of each choice. Reproduced when the mean r is positive for both types in every mouse,
the excitatory and inhibitory means do not differ in any mouse (unpaired t-test across sessions,
p < 0.05), and mouse 4 has the lowest mean r for at least one type. 16 Hz trials are left out of
both, as for the ROC.

In [ ]:
rows = []
for m in MICE:
    rr_ = [r for r in RO if r["mouse"] == m and "auc_err" in r and min(r["n_err"]) >= 10]
    re_ = np.array([stats.pearsonr(r["auc_corr"]["exc"], r["auc_err"]["exc"])[0] for r in rr_])
    ri_ = np.array([stats.pearsonr(r["auc_corr"]["inh"], r["auc_err"]["inh"])[0] for r in rr_])
    tt = nj.ttest(re_, ri_)
    rows.append(dict(mouse=m, sessions=len(rr_), r_exc=np.nanmean(re_) if len(re_) else np.nan,
                     r_inh=np.nanmean(ri_) if len(ri_) else np.nan, p=tt[1]))
t = pd.DataFrame(rows)
print(t.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("Sessions with >= 10 error trials of each choice:", int(t.sessions.sum()), "of", len(RO))
print("Mean r positive for both types in every mouse:", bool(((t.r_exc > 0) & (t.r_inh > 0)).all()))
print("Mice where E and I differ (p < 0.05):", t.mouse[t.p < 0.05].tolist())
print("Mouse with lowest mean r: excitatory", int(t.mouse[t.r_exc.idxmin()]), ", inhibitory", int(t.mouse[t.r_inh.idxmin()]))

**Result: (to be written)**

## fig6a-same-selectivity-correlations

> Noise correlations are stronger between neurons with the same choice selectivity than between
> neurons with opposite selectivity, for excitatory-inhibitory, excitatory-excitatory and
> inhibitory-inhibitory pairs, significant in every mouse except mouse 3 for EE and II pairs; the
> same holds in the spontaneous pre-trial period.

Criterion: preference by the sign of AUC - 0.5 at 0-97 ms before the choice (correct trials);
noise correlations at the same bin on trials with a choice after subtracting each choice's mean;
each neuron's r averaged over its partners of the given type and preference; spontaneous at 0-97 ms
before the initiation tone without subtraction. Test: unpaired t-test same against opposite on
neurons pooled over sessions (code); across-session test as a variant. Reproduced when same exceeds
opposite significantly for EI pairs in all mice and for EE and II pairs in mice 1, 2 and 4, for noise
and spontaneous correlations; partially reproduced when this holds for at least two of the three
pair types. For EI pairs the claims file does not say whose partners are averaged; both readings
are run (EI_e: each excitatory neuron over its inhibitory partners; EI_i: each inhibitory neuron
over its excitatory partners), and a question is asked.

In [ ]:
def pref_table(kind):
    rows = []
    for m in MICE:
        rr_ = [r for r in RO if r["mouse"] == m and kind in r.get("corr_pref", {})]
        for pair in ("EE", "II", "EI_e", "EI_i"):
            same = np.concatenate([r["corr_pref"][kind][pair][0] for r in rr_])
            opp = np.concatenate([r["corr_pref"][kind][pair][1] for r in rr_])
            tp = nj.ttest(same, opp)
            ss = np.array([np.nanmean(r["corr_pref"][kind][pair][0]) for r in rr_])
            so = np.array([np.nanmean(r["corr_pref"][kind][pair][1]) for r in rr_])
            tsess = nj.ttest(ss, so)
            rows.append(dict(kind=kind, mouse=m, pair=pair, same=np.nanmean(same), opposite=np.nanmean(opp),
                             n_same=tp[2], n_opp=tp[3], p_pooled=tp[1], p_sessions=tsess[1]))
    return pd.DataFrame(rows)
T6 = pd.concat([pref_table("noise"), pref_table("spont")])
print(T6.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
def judge(t, ptest):
    sig = (t.same > t.opposite) & (t[ptest] < 0.05)
    out = {}
    for ei in ("EI_e", "EI_i"):
        ok_ei = all(sig[(t.pair == ei) & (t.mouse == m)].iloc[0] for m in MICE)
        ok_ee = all(sig[(t.pair == "EE") & (t.mouse == m)].iloc[0] for m in (1, 2, 4))
        ok_ii = all(sig[(t.pair == "II") & (t.mouse == m)].iloc[0] for m in (1, 2, 4))
        out[ei] = (ok_ei, ok_ee, ok_ii)
    return out
for kind in ("noise", "spont"):
    for ptest in ("p_pooled", "p_sessions"):
        print(kind, ptest, "pattern holds (EI, EE, II) for EI reading:", judge(T6[T6.kind == kind], ptest))

**Result: (to be written)**

## fig6bc-inhibitory-correlations-higher

> Inhibitory pairs have significantly higher noise correlations than excitatory pairs, with
> intermediate values for excitatory-inhibitory pairs, also for spontaneous correlations and for
> neurons matched in spiking activity.

Criterion: per-neuron noise correlation as in fig6a averaged over partners of the same type (EE,
II) or the other type (EI); spontaneous as there; matched activity by the 50-bin max-bin rule.
Unpaired two-tailed t-test, readings (a) across sessions per mouse on session means and (b) on
neurons pooled over sessions. Reproduced when II exceeds EE significantly in every mouse for noise,
spontaneous and matched-activity correlations, with EI between them in the mean; the expected counts
(11,867 excitatory and 1,583 inhibitory neurons in the example mouse) are matched by one mouse.
Matched activity is tested for noise (S10D) and spontaneous (S10E) correlations.

In [ ]:
rows = []
for kind in ("noise", "spont", "matched_noise", "matched_spont"):
    for m in MICE:
        rr_ = [r for r in R if r["mouse"] == m and kind in r["corr"]]
        v = {p: [r["corr"][kind][p] for r in rr_] for p in ("EE", "II", "EI_e", "EI_i")}
        sm_ = {p: np.array([np.nanmean(x) if np.isfinite(x).any() else np.nan for x in v[p]]) for p in v}
        pl = {p: np.concatenate(v[p]) for p in v}
        ta, tb = nj.ttest(sm_["II"], sm_["EE"]), nj.ttest(pl["II"], pl["EE"])
        rows.append(dict(kind=kind, mouse=m, sessions=len(rr_), EE=np.nanmean(sm_["EE"]), II=np.nanmean(sm_["II"]),
                         EI_e=np.nanmean(sm_["EI_e"]), EI_i=np.nanmean(sm_["EI_i"]), p_a_sessions=ta[1], p_b_neurons=tb[1],
                         n_exc=int(np.isfinite(pl["EE"]).sum()), n_inh=int(np.isfinite(pl["II"]).sum())))
t = pd.DataFrame(rows)
t["EI_e_between"] = (t.EI_e > t[["EE", "II"]].min(axis=1)) & (t.EI_e < t[["EE", "II"]].max(axis=1))
t["EI_i_between"] = (t.EI_i > t[["EE", "II"]].min(axis=1)) & (t.EI_i < t[["EE", "II"]].max(axis=1))
print(t.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
for rd in ("p_a_sessions", "p_b_neurons"):
    ok = (t.II > t.EE) & (t[rd] < 0.05)
    print(f"II > EE significant in every mouse and kind, reading {rd}: {bool(ok.all())}; failing rows:",
          t.loc[~ok, ["kind", "mouse"]].values.tolist())
print("EI between EE and II in every row: EI_e", bool(t.EI_e_between.all()), ", EI_i", bool(t.EI_i_between.all()))
nn = t[t.kind == "noise"][["mouse", "n_exc", "n_inh"]]
nn["exc_within"] = (nn.n_exc - 11867).abs() <= 1187; nn["inh_within"] = (nn.n_inh - 1583).abs() <= 158
print(nn.to_string(index=False))

**Result: (to be written)**

## fig8f-correlations-decrease

> Pairwise noise correlations are higher early in training than late, for all combinations of
> neuron types.

Criterion: noise correlations as fig6bc (EE, II, EI), early and late days as fig8de (performance,
the fraction correct on easy trials, below the 20th or above the 80th percentile of the mouse's days;
readings (a) every such day, (b) only the runs at the start and end of training); unpaired
two-tailed t-test across days per mouse at p < 0.05. Reproduced when early exceeds late
significantly for all three pair types in every learning mouse (mice 1, 2 and 4); partially when
early exceeds late in the mean for every type and mouse.

In [ ]:
EL = {}
for m in LEARN:
    d = sess[sess.mouse == m].sort_values("date")
    EL[m] = (d.idx.values, nj.early_late(d.date.values, d.performance.values))
    th = EL[m][1]["thresholds"]
    print(f"mouse {m}: {len(d)} days, performance {d.performance.min():.2f} to {d.performance.max():.2f}, "
          f"20th/80th percentiles {th[0]:.3f}/{th[1]:.3f}; early/late days (a) {EL[m][1]['a'][0].sum()}/{EL[m][1]['a'][1].sum()}, "
          f"(b) {EL[m][1]['b'][0].sum()}/{EL[m][1]['b'][1].sum()}")
byidx = {r["idx"]: r for r in R}
rows = []
for rd in ("a", "b"):
    for m in LEARN:
        ids, el = EL[m]
        e_ids, l_ids = ids[el[rd][0]], ids[el[rd][1]]
        for pair in ("EE", "II", "EI_e", "EI_i"):
            ev_ = np.array([np.nanmean(byidx[i]["corr"]["noise"][pair]) for i in e_ids])
            lv_ = np.array([np.nanmean(byidx[i]["corr"]["noise"][pair]) for i in l_ids])
            tt = nj.ttest(ev_, lv_)
            rows.append(dict(reading=rd, mouse=m, pair=pair, early=np.nanmean(ev_), late=np.nanmean(lv_),
                             n_early=tt[2], n_late=tt[3], p=tt[1]))
t8f = pd.DataFrame(rows)
print(t8f.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
for rd in ("a", "b"):
    for ei in ("EI_e", "EI_i"):
        sub = t8f[(t8f.reading == rd) & t8f.pair.isin(["EE", "II", ei])]
        print(f"reading {rd}, {ei}: early > late significant in all {len(sub)}: {bool(((sub.early > sub.late) & (sub.p < 0.05)).all())}; "
              f"early > late in the mean in all: {bool((sub.early > sub.late).all())}")

**Result: (to be written)**

## fig8g-fraction-selective-triples

> The fraction of choice-selective neurons increases about threefold from early to late training,
> in both excitatory and inhibitory neurons.

Criterion: ROC at 0-97 ms before the choice; early and late days as fig8de; per mouse the mean
fraction over early and over late days. Ratio read (a) per mouse and type, then the mean over mice,
and (b) on fractions pooled over the three mice. Reproduced when, for both types, the ratio is
within tolerance (3 +/- 0.3) under one reading and late exceeds early by the unpaired two-tailed
t-test across days in every learning mouse; partially reproduced when the increase is significant
but the ratio outside tolerance. Sessions without the ROC (fewer than 10 trials of a choice) are
left out.

In [ ]:
rows = []
for rd in ("a", "b"):
    for typ in ("exc", "inh"):
        E_all, L_all = [], []
        for m in LEARN:
            ids, el = EL[m]
            fe = [byidx[i]["roc"][typ]["sig"][byidx[i]["i1"]].mean() for i in ids[el[rd][0]] if byidx[i]["roc_ok"]]
            fl = [byidx[i]["roc"][typ]["sig"][byidx[i]["i1"]].mean() for i in ids[el[rd][1]] if byidx[i]["roc_ok"]]
            tt = nj.ttest(fl, fe)
            rows.append(dict(days=rd, type=typ, mouse=m, early=np.mean(fe) if fe else np.nan, late=np.mean(fl) if fl else np.nan,
                             n_early=len(fe), n_late=len(fl), ratio=(np.mean(fl) / np.mean(fe)) if fe and np.mean(fe) > 0 else np.nan, p=tt[1]))
            E_all += fe; L_all += fl
        rows.append(dict(days=rd, type=typ, mouse="pooled", early=np.mean(E_all), late=np.mean(L_all), n_early=len(E_all),
                         n_late=len(L_all), ratio=np.mean(L_all) / np.mean(E_all), p=np.nan))
t8g = pd.DataFrame(rows)
print(t8g.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
for rd in ("a", "b"):
    for typ in ("exc", "inh"):
        s_ = t8g[(t8g.days == rd) & (t8g.type == typ)]
        per = s_[s_.mouse != "pooled"]
        ra, rb = per.ratio.mean(), s_[s_.mouse == "pooled"].ratio.iloc[0]
        sig = bool(((per.late > per.early) & (per.p < 0.05)).all())
        print(f"days {rd}, {typ}: ratio (a) mean over mice {ra:.2f}, (b) pooled {rb:.2f}; within 2.7-3.3: "
              f"{abs(ra - 3) <= 0.3}/{abs(rb - 3) <= 0.3}; late > early significant in every mouse: {sig}")

**Result: (to be written)**

## figs2-temporal-dynamics-similar

> Excitatory and inhibitory neurons have similar temporal dynamics, with similar distributions of
> peak-activity times across the trial.

Criterion: per session the fraction of neurons of each type whose trial-averaged activity peaks in
each 100 ms window, traces aligned to each trial event (start tone, stimulus, choice, reward) and
concatenated, with the windows of the dataset's trial-segmented responses; unpaired two-tailed
t-test across sessions per mouse in each window, a claim of absence; readings (a) uncorrected
p < 0.05, with the number of differing windows no more than chance by a binomial test at 0.05, and
(b) none differing after Bonferroni. Correct trials are averaged (the reward-aligned traces exist
only for them); each event's window is cut to the frames all of a mouse's sessions share, so that
windows line up across sessions.

In [ ]:
rows = []
for m in MICE:
    rr_ = [r for r in R if r["mouse"] == m and r["figs2"] is not None]
    keys = ("init", "stim", "choice", "reward")
    before = {k: min(r["figs2"]["events"][k] for r in rr_) for k in keys}
    after = {k: min(r["figs2"]["segs"][k].shape[0] - r["figs2"]["events"][k] for r in rr_) for k in keys}
    F = [S.peak_window_fractions(r["figs2"], before, after) for r in rr_]
    fe = np.stack([f["exc"] for f in F]); fi = np.stack([f["inh"] for f in F])
    p = np.array([stats.ttest_ind(fe[:, k], fi[:, k]).pvalue if np.std(np.r_[fe[:, k], fi[:, k]]) > 0 else 1.0
                  for k in range(fe.shape[1])])
    nw = len(p); nsig = int((p < 0.05).sum())
    binom = stats.binomtest(nsig, nw, 0.05, alternative="greater").pvalue
    rows.append(dict(mouse=m, sessions=len(rr_), windows=nw, frames_per_event=str({k: before[k] + after[k] for k in keys}),
                     windows_p05=nsig, binomial_p=binom, windows_bonferroni=int((p < 0.05 / nw).sum()), min_p=p.min()))
t = pd.DataFrame(rows)
print(t.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("Reading (a), no more differing windows than chance in every mouse:", bool((t.binomial_p >= 0.05).all()))
print("Reading (b), no window differs after Bonferroni in any mouse:", bool((t.windows_bonferroni == 0).all()))

**Result: (to be written)**

## figs5-fluorescence-decoding, figs14-neuropil, figs15-labeling-specificity (blocked)

- figs5 needs the CNMF denoised fluorescence traces or Delta F/F traces. The files hold one kind of
  trace, the inferred spiking activity (data check above), under groups named `dFoF_*`.
- figs14 needs the raw imaging movies and the CNMF spatial footprints; the files hold the ROI masks
  but no movie.
- figs15 needs the immunostained sections; the dandiset holds imaging sessions only.

The paper's own deposit at the CSHL repository (record 36980, a 70 GB split gzip archive) is said
to hold the raw data and the CNMF outputs; the claims file does not name it as data the claims use,
so it is not read here, and a question is asked.

In [ ]:
print("Processing modules and acquisition groups in the files:", sorted(set(k for r in R for k in r["available"])))

## Summary

(to be written)